# 🔵 DATA NAPOLI — Mega-Database 1926–2026

**Obiettivo:** Estrarre OGNI partita ufficiale della SSC Napoli dal 1° agosto 1926 al 1° ottobre 2026, fondendo dati da 5 fonti indipendenti.

| Fonte | Dati | Copertura | Metodo |
|---|---|---|---|
| **10maggio87.it** | Risultati, marcatori, rosa, allenatore | 1926–oggi | `__NEXT_DATA__` JSON (no browser) |
| **Transfermarkt** | Formazioni, sostituzioni, arbitri, stadi | ~1963–oggi | requests + BS4 |
| **StadiaPostcards** | Presenze allo stadio partita per partita | Serie A 1963–oggi | requests + BS4 |
| **FBref** | xG, possesso, tiri, PPDA, passaggi, contrasti | 2017–oggi | soccerdata |
| **Understat** | xG per tiro, coordinate shot map | 2014–oggi | understat API |

**Regola d'oro:** Zero dati inventati. Se una fonte non ha un campo, il campo è `null`.

**Salvataggio incrementale:** dopo ogni stagione, i dati vengono salvati su disco. Se Colab si disconnette, riparti da dove eri rimasto.

## 1. Installazione dipendenze

In [ ]:
!pip install requests beautifulsoup4 lxml pandas soccerdata understat nest_asyncio

import requests
import json
import re
import os
import time
import asyncio
import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

nest_asyncio.apply()

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Cartella di lavoro su Colab
OUTPUT_DIR = '/content/napoli_db'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup completato.')

## 2. Generazione lista stagioni (1926-27 → 2026-27)

In [ ]:
ALL_SEASONS = [f'{y}-{str(y+1)[-2:].zfill(2)}' for y in range(1926, 2027)]
print(f'Stagioni da estrarre: {len(ALL_SEASONS)}')
print(f'Prima: {ALL_SEASONS[0]}  —  Ultima: {ALL_SEASONS[-1]}')

## 3. Dizionario di Continuità Societaria (Lineage)

In [ ]:
CLUB_LINEAGE = {
    # Fusioni / assorbimenti storici verificati
    'Alba Roma':           {'modern_id': 'roma',      'modern_name': 'AS Roma',       'note': 'Confluita nella Roma nel 1927. Albo unificato.'},
    'Fortitudo Roma':      {'modern_id': 'roma',      'modern_name': 'AS Roma',       'note': 'Confluita nella Roma nel 1927.'},
    'Roman FC':            {'modern_id': 'roma',      'modern_name': 'AS Roma',       'note': 'Confluita nella Roma nel 1927.'},
    'Ambrosiana':          {'modern_id': 'inter',     'modern_name': 'Inter',         'note': 'Nome assunto dall\'Inter 1928-1945. Albo unificato.'},
    'Ambrosiana-Inter':    {'modern_id': 'inter',     'modern_name': 'Inter',         'note': 'Nome assunto dall\'Inter 1928-1945. Albo unificato.'},
    'Sampierdarenese':     {'modern_id': 'sampdoria', 'modern_name': 'Sampdoria',     'note': 'Confluita nella Sampdoria nel 1946.'},
    'Andrea Doria':        {'modern_id': 'sampdoria', 'modern_name': 'Sampdoria',     'note': 'Confluita nella Sampdoria nel 1946.'},
    'Liguria':             {'modern_id': 'sampdoria', 'modern_name': 'Sampdoria',     'note': 'Nome di guerra della fusione Sampierdarenese-Andrea Doria (1943-46).'},
    'Vigor':               {'modern_id': 'sampdoria', 'modern_name': 'Sampdoria',     'note': 'Antenata della Sampierdarenese.'},
    'MATER Roma':          {'modern_id': 'roma',      'modern_name': 'AS Roma',       'note': 'Squadra operaia romana, assorbita.'},
    'Pro Vercelli':        {'modern_id': 'pro_vercelli', 'modern_name': 'Pro Vercelli', 'note': 'Club ancora esistente. Nessun assorbimento.'},
    'Casale':              {'modern_id': 'casale',    'modern_name': 'Casale FBC',    'note': 'Club ancora esistente nelle serie minori.'},
    'Fiorentina':          {'modern_id': 'fiorentina','modern_name': 'ACF Fiorentina','note': 'Fallimento 2002, rifondata. FIGC riconosce continuità.'},
    'SSC Napoli':          {'modern_id': 'napoli',    'modern_name': 'SSC Napoli',    'note': 'Fallimento 2004, rifondato come Napoli Soccer. FIGC riconosce continuità dal 1926.'},
    'Parma AC':            {'modern_id': 'parma',     'modern_name': 'Parma Calcio 1913', 'note': 'Fallimento 2015, rifondato. FIGC riconosce continuità.'},
}

def resolve_opponent(name):
    clean = name.strip()
    if clean in CLUB_LINEAGE:
        entry = CLUB_LINEAGE[clean]
        return {
            'historical_name': clean,
            'modern_id': entry['modern_id'],
            'modern_name': entry['modern_name'],
            'lineage_note': entry['note']
        }
    return {
        'historical_name': clean,
        'modern_id': clean.lower().replace(' ', '_'),
        'modern_name': clean,
        'lineage_note': None
    }

print(f'Lineage: {len(CLUB_LINEAGE)} club storici mappati.')

## 4. FONTE 1 — 10maggio87.it (Backbone storico)
Estrae il blob JSON `__NEXT_DATA__` dalla pagina. Contiene TUTTI i dati pre-renderizzati: partite, marcatori, rosa, allenatore. **Nessun browser necessario.**

In [ ]:
def scrape_10maggio87(season):
    """Estrae __NEXT_DATA__ da 10maggio87.it per una stagione."""
    url = f'https://www.10maggio87.it/Almanacco/Napoli/{season}/'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            print(f'  [10m87] HTTP {resp.status_code} per {season}')
            return None

        # Estrai il blob __NEXT_DATA__
        soup = BeautifulSoup(resp.text, 'lxml')
        script_tag = soup.find('script', id='__NEXT_DATA__')
        if not script_tag:
            print(f'  [10m87] Nessun __NEXT_DATA__ trovato per {season}')
            return None

        next_data = json.loads(script_tag.string)
        page_props = next_data.get('props', {}).get('pageProps', {})

        # Esplora la struttura dei dati
        season_info = {
            'season': season,
            'source': '10maggio87',
            'raw_page_props_keys': list(page_props.keys()),
            'players': [],
            'matches': [],
            'managers': [],
            'season_stats': {},
            'competitions': []
        }

        # Estrarre i giocatori dalla rosa
        players_data = page_props.get('players', page_props.get('squad', page_props.get('roster', [])))
        if isinstance(players_data, list):
            for p in players_data:
                if isinstance(p, dict):
                    season_info['players'].append(p)

        # Estrarre le partite (cerchiamo in tutte le chiavi possibili)
        for key in page_props:
            val = page_props[key]
            if isinstance(val, list) and len(val) > 0:
                sample = val[0] if isinstance(val[0], dict) else {}
                sample_keys = set(k.lower() for k in sample.keys()) if sample else set()
                # Riconosciamo una lista di partite se ha chiavi come 'result', 'opponent', 'date', 'score', 'goal'
                match_indicators = {'result', 'opponent', 'date', 'score', 'goals', 'home', 'away', 'match', 'fixture',
                                    'risultato', 'avversario', 'data', 'punteggio', 'gol', 'casa', 'trasferta', 'partita'}
                if sample_keys & match_indicators:
                    for m in val:
                        if isinstance(m, dict):
                            season_info['matches'].append(m)
                    print(f'  [10m87] Trovate {len(val)} partite nella chiave "{key}"')

        # Estrarre manager/allenatore
        managers_data = page_props.get('managers', page_props.get('coaches', page_props.get('staff', [])))
        if isinstance(managers_data, list):
            for mg in managers_data:
                if isinstance(mg, dict):
                    season_info['managers'].append(mg)

        # Statistiche stagionali aggregate
        for key in ['stats', 'standings', 'classifica', 'statistics', 'resume']:
            if key in page_props and page_props[key]:
                season_info['season_stats'][key] = page_props[key]

        # Competizioni
        for key in ['competitions', 'competizioni', 'seasons']:
            if key in page_props and isinstance(page_props[key], list):
                season_info['competitions'] = page_props[key]

        # Se non abbiamo trovato partite con euristica sulle chiavi, salviamo TUTTO il pageProps
        # per non perdere nulla (Zero Loss)
        if not season_info['matches']:
            season_info['_raw_page_props_dump'] = page_props
            print(f'  [10m87] Nessuna chiave partite riconosciuta. Dump completo salvato. Chiavi: {list(page_props.keys())}')

        print(f'  [10m87] {season}: {len(season_info["players"])} giocatori, {len(season_info["matches"])} partite, {len(season_info["managers"])} allenatori')
        return season_info

    except Exception as e:
        print(f'  [10m87] Errore per {season}: {e}')
        return None

# Test su 2 stagioni di prova
print('=== TEST 10maggio87 ===')
test1 = scrape_10maggio87('1986-87')
time.sleep(4)
test2 = scrape_10maggio87('2024-25')

if test1:
    print(f'\n1986-87 chiavi pageProps: {test1["raw_page_props_keys"]}')
if test2:
    print(f'2024-25 chiavi pageProps: {test2["raw_page_props_keys"]}')

## 5. FONTE 2 — Transfermarkt (Formazioni, sostituzioni, stadi, arbitri)
HTML server-side rendering. `requests + BeautifulSoup`.

In [ ]:
TM_BASE = 'https://www.transfermarkt.it/ssc-napoli/spielplandatum/verein/6195/plus/1'

def scrape_transfermarkt_season(season):
    """Estrae risultati e tabellini da Transfermarkt per una stagione."""
    # Transfermarkt usa l'anno di inizio della stagione nell'URL
    year_start = int(season.split('-')[0])
    url = f'{TM_BASE}?saession_id={year_start}'
    
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            return None

        soup = BeautifulSoup(resp.text, 'lxml')
        matches = []

        # Transfermarkt ha tabelle con classe 'items'
        rows = soup.select('table.items tbody tr')
        for row in rows:
            cells = row.find_all('td')
            if len(cells) >= 5:
                match_entry = {
                    'source': 'transfermarkt',
                    'cells_text': [c.get_text(strip=True) for c in cells]
                }
                # Estrai link al tabellino per dettagli (formazione, subs, cartellini)
                link = row.find('a', href=re.compile(r'/spielbericht/'))
                if link:
                    match_entry['detail_url'] = 'https://www.transfermarkt.it' + link['href']
                matches.append(match_entry)

        print(f'  [TM] {season}: {len(matches)} partite trovate')
        return {'season': season, 'source': 'transfermarkt', 'matches': matches}

    except Exception as e:
        print(f'  [TM] Errore per {season}: {e}')
        return None

def scrape_tm_match_detail(detail_url):
    """Estrae formazione, sostituzioni, cartellini, stadio e arbitro da una singola partita TM."""
    try:
        resp = requests.get(detail_url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, 'lxml')
        detail = {'source': 'transfermarkt', 'url': detail_url}

        # Stadio
        stadium_el = soup.find('a', href=re.compile(r'/stadion/'))
        detail['stadium'] = stadium_el.get_text(strip=True) if stadium_el else None

        # Arbitro
        referee_el = soup.find('a', href=re.compile(r'/schiedsrichter/'))
        detail['referee'] = referee_el.get_text(strip=True) if referee_el else None

        # Presenze
        att_el = soup.find('td', string=re.compile(r'Spettatori', re.I))
        if att_el:
            val = att_el.find_next('td')
            if val:
                num = re.sub(r'[^\d]', '', val.get_text())
                detail['attendance'] = int(num) if num else None

        # Formazioni (titolari)
        lineups = []
        lineup_divs = soup.select('div.aufstellung-spieler-container')
        for div in lineup_divs:
            player_links = div.select('a.aufstellung-spieler-name')
            for pl in player_links:
                lineups.append(pl.get_text(strip=True))
        detail['lineups'] = lineups if lineups else None

        # Cartellini
        cards = []
        card_icons = soup.select('span.icon-gelbekarte, span.icon-rotekarte, span.icon-gelbrotekarte')
        for icon in card_icons:
            parent = icon.find_parent('div')
            if parent:
                cards.append(parent.get_text(strip=True))
        detail['cards'] = cards if cards else None

        return detail

    except Exception as e:
        return None

# Test
print('=== TEST Transfermarkt ===')
tm_test = scrape_transfermarkt_season('2024-25')
if tm_test and tm_test['matches']:
    print(f'  Prima partita raw: {tm_test["matches"][0]["cells_text"][:5]}')

## 6. FONTE 3 — StadiaPostcards (Presenze allo stadio dal 1963)

In [ ]:
def scrape_stadiapostcards(season):
    """Estrae le presenze dalla pagina della stagione su StadiaPostcards."""
    # Formato URL: A26-27.htm per Serie A 2026-27
    parts = season.split('-')
    suffix = f'{parts[0][-2:]}-{parts[1]}'
    url = f'https://www.stadiapostcards.com/A{suffix}.htm'

    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        if resp.status_code != 200:
            return None

        # StadiaPostcards è HTML puro anni 2000 — Pandas lo legge perfettamente
        tables = pd.read_html(resp.text, flavor='lxml')
        attendance_data = []

        for df in tables:
            df_str = df.astype(str).to_string().lower()
            # Cerca tabelle che contengono 'napoli' e numeri di spettatori
            if 'napoli' in df_str:
                for _, row in df.iterrows():
                    row_dict = row.dropna().to_dict()
                    row_str = str(row_dict).lower()
                    if 'napoli' in row_str:
                        attendance_data.append({
                            'source': 'stadiapostcards',
                            'raw': row_dict
                        })

        print(f'  [SP] {season}: {len(attendance_data)} righe con Napoli trovate')
        return {'season': season, 'source': 'stadiapostcards', 'attendance': attendance_data}

    except Exception as e:
        return None

# Test (solo stagioni recenti — pre-1963 non esiste)
print('=== TEST StadiaPostcards ===')
sp_test = scrape_stadiapostcards('2024-25')

## 7. FONTE 4 — FBref via soccerdata (Stats avanzate dal 2017)

In [ ]:
def scrape_fbref_season(season):
    """Estrae statistiche avanzate da FBref tramite soccerdata."""
    year_start = int(season.split('-')[0])
    if year_start < 2017:
        return None  # FBref non ha dati pre-2017

    try:
        import soccerdata as sd
        fbref = sd.FBref(leagues='ITA-Serie A', seasons=year_start)

        # Statistiche di squadra
        team_stats = fbref.read_team_season_stats(stat_type='standard')
        napoli_stats = team_stats.xs('Napoli', level='team', drop_level=False) if 'Napoli' in team_stats.index.get_level_values('team') else None

        # Risultati partita per partita
        schedule = fbref.read_schedule()
        napoli_matches = schedule[
            (schedule.index.get_level_values('home_team') == 'Napoli') |
            (schedule.index.get_level_values('away_team') == 'Napoli')
        ] if schedule is not None else None

        result = {
            'season': season,
            'source': 'fbref',
            'team_stats': napoli_stats.to_dict() if napoli_stats is not None else None,
            'matches': napoli_matches.reset_index().to_dict(orient='records') if napoli_matches is not None else []
        }
        print(f'  [FBref] {season}: {len(result["matches"])} partite')
        return result

    except Exception as e:
        print(f'  [FBref] {season}: {e}')
        return None

print('=== TEST FBref ===')
print('  (FBref ha rate limit rigido — test singolo)')
# fbref_test = scrape_fbref_season('2023-24')  # Decommentare per testare

## 8. FONTE 5 — Understat (xG e coordinate tiri dal 2014)

In [ ]:
async def scrape_understat_season(season):
    """Estrae xG partita per partita da Understat."""
    year_start = int(season.split('-')[0])
    if year_start < 2014:
        return None  # Understat non ha dati pre-2014

    try:
        from understat import Understat
        async with Understat() as us:
            results = await us.get_team_results('Napoli', year_start)
            matches = []
            for r in results:
                matches.append({
                    'source': 'understat',
                    'date': r.get('datetime', ''),
                    'home': r.get('h', {}).get('title', ''),
                    'away': r.get('a', {}).get('title', ''),
                    'home_goals': r.get('goals', {}).get('h', None),
                    'away_goals': r.get('goals', {}).get('a', None),
                    'xG_home': r.get('xG', {}).get('h', None),
                    'xG_away': r.get('xG', {}).get('a', None),
                    'match_id': r.get('id', None)
                })
            print(f'  [US] {season}: {len(matches)} partite con xG')
            return {'season': season, 'source': 'understat', 'matches': matches}

    except Exception as e:
        print(f'  [US] {season}: {e}')
        return None

print('=== TEST Understat ===')
us_test = asyncio.get_event_loop().run_until_complete(scrape_understat_season('2023-24'))

## 9. 🚀 ORCHESTRATORE PRINCIPALE — Fusione multi-fonte
Esegue lo scraping completo di tutte le 101 stagioni, salvando incrementalmente dopo ogni stagione.

In [ ]:
def load_checkpoint():
    """Carica il database parziale se esiste (resilienza)."""
    path = os.path.join(OUTPUT_DIR, 'checkpoint.json')
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'Checkpoint caricato: {len(data["seasons_completed"])} stagioni già estratte.')
        return data
    return {'seasons_completed': [], 'database': []}

def save_checkpoint(state):
    """Salva il database parziale dopo ogni stagione."""
    path = os.path.join(OUTPUT_DIR, 'checkpoint.json')
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False)

def run_full_scrape():
    state = load_checkpoint()
    done = set(state['seasons_completed'])
    remaining = [s for s in ALL_SEASONS if s not in done]

    print(f'\n{"="*60}')
    print(f'MEGA-DATABASE NAPOLI — Avvio scraping')
    print(f'Stagioni totali: {len(ALL_SEASONS)}')
    print(f'Già completate: {len(done)}')
    print(f'Da fare: {len(remaining)}')
    print(f'{"="*60}\n')

    for i, season in enumerate(remaining):
        print(f'\n[{i+1}/{len(remaining)}] === STAGIONE {season} ===')
        year_start = int(season.split('-')[0])

        season_record = {
            'season': season,
            'scraped_at': datetime.now().isoformat(),
            'fonte_10maggio87': None,
            'fonte_transfermarkt': None,
            'fonte_stadiapostcards': None,
            'fonte_fbref': None,
            'fonte_understat': None
        }

        # FONTE 1: 10maggio87 (tutte le stagioni)
        season_record['fonte_10maggio87'] = scrape_10maggio87(season)
        time.sleep(4)  # Delay di rispetto

        # FONTE 2: Transfermarkt (dal ~1963 in poi, ma tenta tutto)
        season_record['fonte_transfermarkt'] = scrape_transfermarkt_season(season)
        time.sleep(5)

        # FONTE 3: StadiaPostcards (Serie A dal 1963)
        if year_start >= 1963:
            season_record['fonte_stadiapostcards'] = scrape_stadiapostcards(season)
            time.sleep(2)

        # FONTE 4: FBref (dal 2017)
        if year_start >= 2017:
            season_record['fonte_fbref'] = scrape_fbref_season(season)
            time.sleep(4)

        # FONTE 5: Understat (dal 2014)
        if year_start >= 2014:
            season_record['fonte_understat'] = asyncio.get_event_loop().run_until_complete(
                scrape_understat_season(season)
            )
            time.sleep(2)

        # Risolvi lineage per ogni avversario trovato
        if season_record['fonte_10maggio87'] and season_record['fonte_10maggio87'].get('matches'):
            for match in season_record['fonte_10maggio87']['matches']:
                opp_name = match.get('opponent', match.get('avversario', match.get('away', '')))
                if isinstance(opp_name, str) and opp_name:
                    match['_resolved_opponent'] = resolve_opponent(opp_name)

        # Salvataggio incrementale
        state['database'].append(season_record)
        state['seasons_completed'].append(season)
        save_checkpoint(state)

        # Report progresso
        sources_ok = sum(1 for k in ['fonte_10maggio87','fonte_transfermarkt','fonte_stadiapostcards','fonte_fbref','fonte_understat'] if season_record[k])
        print(f'  ✅ {season} completata ({sources_ok}/5 fonti attive). Totale: {len(state["seasons_completed"])}/{len(ALL_SEASONS)}')

    print(f'\n{"="*60}')
    print(f'SCRAPING COMPLETATO! {len(state["seasons_completed"])} stagioni nel database.')
    print(f'{"="*60}')
    return state

# 🚀 AVVIA LO SCRAPING COMPLETO
final_state = run_full_scrape()

## 10. Export del database finale

In [ ]:
# Esporta il database completo in un singolo file JSON
final_path = os.path.join(OUTPUT_DIR, 'master_database_napoli.json')
with open(final_path, 'w', encoding='utf-8') as f:
    json.dump(final_state['database'], f, indent=2, ensure_ascii=False, default=str)

file_size = os.path.getsize(final_path)
print(f'Database esportato: {final_path}')
print(f'Dimensione: {file_size / (1024*1024):.1f} MB')
print(f'Stagioni: {len(final_state["database"])}')

# Conteggio partite totali
total_matches = 0
for s in final_state['database']:
    for fonte_key in ['fonte_10maggio87', 'fonte_transfermarkt']:
        fonte = s.get(fonte_key)
        if fonte and isinstance(fonte.get('matches'), list):
            total_matches += len(fonte['matches'])
            break  # Conta solo dalla prima fonte disponibile per evitare duplicati
print(f'Partite totali stimate: {total_matches}')

In [ ]:
from google.colab import files
files.download(final_path)